# StoryZop: Instagram Story Visual Analysis

This notebook runs the complete StoryZop pipeline on Google Colab.

**Before running:**
1. Change runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Export your Instagram cookies locally:
   ```bash
   pip install browser-cookie3
   python extract_cookies.py
   ```
3. Upload `data/session.json` to this Colab notebook (use the file upload in Cell 6)

## 1. Install Dependencies & Clone Repo

In [ ]:
# Core dependencies
!pip install -q playwright Pillow pydantic pydantic-settings python-dotenv nest-asyncio

# Install Chromium WITH system dependencies (critical for Colab)
!playwright install --with-deps chromium

# AI / Vision dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers>=4.57.0 accelerate qwen-vl-utils bitsandbytes easyocr

# Clone the repo
!git clone https://github.com/10Unknownboy/StoryZop.git 2>/dev/null || echo 'Repo already cloned'
%cd StoryZop

# Enable async in Colab
import nest_asyncio
nest_asyncio.apply()

print('\n\u2705 All dependencies installed.')

## 2. GPU Check

In [ ]:
from src.vision.gpu import GPUManager

gpu_info = GPUManager.detect_gpu()
GPUManager.print_gpu_status()

print(f"\n4B (4-bit) fits: {GPUManager.estimate_model_fit('4b', quantized=True)}")
print(f"8B (4-bit) fits: {GPUManager.estimate_model_fit('8b', quantized=True)}")
print(f"32B (4-bit) fits: {GPUManager.estimate_model_fit('32b', quantized=True)}")

## 3. Configuration

In [ ]:
from src.config import get_config

config = get_config(
    use_4bit_quantization=True,
    headless=True,
)

print(f'Data dir: {config.data_dir}')
print(f'Models: {config.initial_model} | {config.primary_model}')

## 4. Initialize Database

In [ ]:
from src.database.database import Database

db = Database(config.db_path)
db.initialize()

state = db.get_processing_state()
print(f'\u2705 Database ready at {config.db_path}')
print(f'   Completed: {len(state["completed"])} | Pending: {len(state["pending"])} | Incomplete: {len(state["incomplete"])}')

## 5. Load AI Models

In [ ]:
from src.vision.qwen4b import Qwen4BScreener
from src.vision.qwen8b import Qwen8BAnalyzer
from src.vision.qwen32b import Qwen32BExpert
from src.vision.ocr import OCREngine

screener = Qwen4BScreener(config)
analyzer = Qwen8BAnalyzer(config)

expert = None
if GPUManager.estimate_model_fit('32b', quantized=config.use_4bit_quantization):
    expert = Qwen32BExpert(config)
    print('\u2705 32B Expert enabled.')
else:
    print('\u26a0\ufe0f 32B Expert skipped (not enough VRAM).')

ocr_engine = OCREngine(
    languages=config.ocr_languages,
    confidence_threshold=config.ocr_confidence_threshold
)
print(f'\u2705 OCR engine ready (available: {ocr_engine.is_available})')

print('\nVerifying 4B model access...')
screener.load_model()
print('\u2705 4B loaded successfully!')
screener.unload_model()
print('\u2705 4B unloaded. Models will reload during pipeline run.\n')
print('--- All models ready ---')

## 6. Upload session.json & Authenticate

**Upload your `session.json`** file when prompted. This file contains your Instagram cookies
exported by `extract_cookies.py` from your local machine.

If you don't have it yet, run locally:
```bash
pip install browser-cookie3
python extract_cookies.py
```
Then upload the resulting `data/session.json` file here.

In [ ]:
import os
from pathlib import Path
from src.browser.session import BrowserSession
from src.browser.instagram import InstagramNavigator
from src.browser.stories import StoryNavigator
from src.capture.frame_manager import FrameManager
from src.capture.sampler import StorySampler

# --- Step 1: Get cookies ---
session_json_path = Path('data/session.json')

# Check if session.json already exists (e.g., from a previous upload)
if not session_json_path.exists():
    # Try uploading from Colab
    try:
        from google.colab import files
        print('\U0001f4c2 Please upload your session.json file...')
        uploaded = files.upload()
        if uploaded:
            fname = list(uploaded.keys())[0]
            os.makedirs('data', exist_ok=True)
            with open(session_json_path, 'wb') as f:
                f.write(uploaded[fname])
            print(f'\u2705 Saved {fname} as {session_json_path}')
        else:
            print('\u274c No file uploaded.')
    except ImportError:
        print('Not running on Colab. Place session.json in data/ directory.')

if not session_json_path.exists():
    raise FileNotFoundError(
        f'session.json not found at {session_json_path}. '
        'Run extract_cookies.py locally and upload the file.'
    )

# Show cookie summary (no values!)
import json
with open(session_json_path) as f:
    cookies_preview = json.load(f)
print(f'\nLoaded {len(cookies_preview)} cookies:')
for c in cookies_preview:
    print(f"  {c['name']}")

# --- Step 2: Launch browser and load cookies ---
session = BrowserSession(config)
await session.launch()
print('\n\u2705 Browser launched.')

# Load the full cookie set
await session.load_cookies(session_json_path)
print('\u2705 Cookies loaded.')

# Navigate to Instagram
is_authed = await session.navigate_and_verify()

# Create components
instagram_nav = InstagramNavigator(session.page, config)
story_nav = StoryNavigator(session.page, config)
frame_manager = FrameManager(config)
sampler = StorySampler(config, frame_manager)

# Dismiss popups
await instagram_nav.dismiss_dialogs()

# Verify auth
is_auth = await instagram_nav.verify_authentication()
print(f'\nAuthenticated: {is_auth}')

if is_auth:
    print('\u2705 Ready to scan stories.')
else:
    print('\u274c Authentication FAILED.')
    print('Your cookies may have expired. Re-run extract_cookies.py locally and re-upload.')

## 6b. Debug: Screenshot

In [ ]:
from IPython.display import display, Image as IPImage
import os

os.makedirs(config.data_dir, exist_ok=True)
debug_path = str(config.data_dir / 'debug_feed.png')

await session.page.screenshot(path=debug_path, full_page=False)
print('If you see your Instagram feed = auth works.')
print('If you see "Open Instagram" or login = cookies expired.')
display(IPImage(filename=debug_path, width=412))

## 6c. Test story discovery

In [ ]:
discovered = await instagram_nav.get_stories_tray()
print(f'Found {len(discovered)} stories:')
for s in discovered:
    print(f"  @{s['username']} (index={s['index']})")

if not discovered:
    print('\n\u26a0\ufe0f No stories found.')
    print('Check the screenshot in cell 6b.')

## 7. Run the Full Pipeline

In [ ]:
from src.pipeline import StoryPipeline

pipeline = StoryPipeline(config, db)
pipeline.set_browser(session)
pipeline.set_navigators(instagram_nav, story_nav)
pipeline.set_sampler(sampler, frame_manager)
pipeline.set_models(screener, analyzer, expert)
pipeline.set_ocr(ocr_engine)

print('Starting pipeline...\n')
stats = await pipeline.run()

print(f'\n--- Pipeline Complete ---')
print(f'  Discovered: {stats["discovered"]}')
print(f'  Completed:  {stats["completed"]}')
print(f'  Revisited:  {stats["revisited"]}')
print(f'  Failed:     {stats["failed"]}')

## 8. View Results

In [ ]:
from src.analysis.report import ReportGenerator

report_gen = ReportGenerator(db)
report = report_gen.generate_text_report()
if report:
    print(report)
else:
    print('No stories analyzed yet.')

## 9. Export Data

In [ ]:
import os
os.makedirs(config.data_dir, exist_ok=True)

json_path = config.data_dir / 'export.json'
csv_path = config.data_dir / 'export.csv'

report_gen.export_json(json_path)
report_gen.export_csv(csv_path)

print(f'\u2705 JSON exported to {json_path}')
print(f'\u2705 CSV exported to {csv_path}')

try:
    from google.colab import files
    files.download(str(json_path))
    files.download(str(csv_path))
except ImportError:
    pass

## 10. Cleanup

In [ ]:
await session.close()
print('\u2705 Browser closed.')

if screener.is_loaded:
    screener.unload_model()
if analyzer.is_loaded:
    analyzer.unload_model()
if expert and expert.is_loaded:
    expert.unload_model()
print('\u2705 Models unloaded.')

db.close()
print('\u2705 Database closed.')